### loom文件 gene_symbol转换

In [34]:
import loompy
import pandas as pd
import numpy as np
from pathlib import Path

In [45]:
# 需要转换gene_symbol的loom文件
loom_path = "/data/wuwq/noise/Monod_FLU/ori_loom/covid11_s20.loom" #改这里
mapping_file = "/data2/wuwq/noise/monod_FLU/polyA_ref/mapping.txt"  #不用变
col_gene_name   = "Gene name"
col_ensembl_id  = "Gene stable ID version"
output_loom = str(Path(loom_path).with_stem(Path(loom_path).stem + "_symbol"))
print(f"输入 loom: {loom_path}")
print(f"输出 loom: {output_loom}")
print(f"映射表: {mapping_file}\n")
### 构建映射表
suffix = Path(mapping_file).suffix.lower()
if suffix == ".csv":
    df_map = pd.read_csv(mapping_file)
elif suffix in [".tsv", ".txt"]:
    df_map = pd.read_csv(mapping_file, sep="\t")
elif suffix == ".xlsx":
    df_map = pd.read_excel(mapping_file)
else:
    raise ValueError("不支持的文件格式，请使用 .csv / .tsv / .xlsx")
print("映射表示例（前5行）:")
print(df_map.head())
print("实际列名列表:")
print(df_map.columns.tolist())
required_cols = [col_ensembl_id, col_gene_name]
missing = [col for col in required_cols if col not in df_map.columns]
if missing:
    raise KeyError(f"映射表中缺少以下列: {missing}\n实际列名: {df_map.columns.tolist()}")
# 清理并构建字典：ensembl_id → gene_name
df_map = df_map[[col_ensembl_id, col_gene_name]].dropna().copy()
df_map[col_ensembl_id] = df_map[col_ensembl_id].astype(str).str.strip()
df_map[col_gene_name]  = df_map[col_gene_name].astype(str).str.strip()
mapping_dict = df_map.set_index(col_ensembl_id)[col_gene_name].to_dict()
print(f"成功构建映射字典，共 {len(mapping_dict):,} 个 Ensembl ID → gene_name\n")

输入 loom: /data/wuwq/noise/Monod_FLU/ori_loom/covid11_s20.loom
输出 loom: /data/wuwq/noise/Monod_FLU/ori_loom/covid11_s20_symbol.loom
映射表: /data2/wuwq/noise/monod_FLU/polyA_ref/mapping.txt

映射表示例（前5行）:
  Gene stable ID version Gene name
0      ENSG00000210049.1     MT-TF
1      ENSG00000211459.2   MT-RNR1
2      ENSG00000210077.1     MT-TV
3      ENSG00000210082.2   MT-RNR2
4      ENSG00000209082.1    MT-TL1
实际列名列表:
['Gene stable ID version', 'Gene name']
成功构建映射字典，共 48,698 个 Ensembl ID → gene_name



In [46]:
### 执行基因名转换
import loompy
import numpy as np
import os

loom_path = "/data/wuwq/noise/Monod_FLU/ori_loom/covid11_s20.loom"      # ← 改：转换前路径
output_loom = "/data/wuwq/noise/Monod_FLU/symbol_loom/covid11_s20.loom"  # ← 改：新文件路径

with loompy.connect(loom_path) as ds:
    print("原始 loom 形状:", ds.shape)
    print("可用层:", list(ds.layers.keys()))
    print("行属性键名:", list(ds.ra.keys()))   
    # 定义正确的键名
    gene_id_key = 'target_name'    
    if gene_id_key not in ds.ra.keys():
        raise KeyError(f"行属性中未找到 '{gene_id_key}'，实际键名: {list(ds.ra.keys())}")   
    # ← 这里修正：原来错误的 ds.ra[target_name] 改为 ds.ra[gene_id_key]
    print("当前前8个基因（Ensembl ID 等）:", ds.ra[gene_id_key][:8])  
    # 当前基因 ID 列表
    current_genes = np.asarray(ds.ra[gene_id_key]).astype(str)    
    new_genes = []
    unmapped_count = 0    
    for g in current_genes:
        symbol = mapping_dict.get(g, None)   # ← 确保你已经定义了 mapping_dict
        if symbol is None or symbol in ["", "nan"]:
            unmapped_count += 1
            new_genes.append(g)              # 未匹配的保留原 ID
        else:
            new_genes.append(symbol)   
    new_genes = np.array(new_genes)   
    print(f"未匹配的基因数量：{unmapped_count} / {len(current_genes)} ({unmapped_count/len(current_genes):.2%})")
    print("替换后前8个基因（gene symbol）:", new_genes[:8]) 
    # 复制行属性并新增标准字段
    row_attrs = {k: v[:] for k, v in ds.ra.items()}
    row_attrs["Gene"] = new_genes          # 大多数工具期待的 gene symbol
    row_attrs["Accession"] = current_genes # 保留原始 Ensembl/target ID   
    col_attrs = {k: v[:] for k, v in ds.ca.items()}
    # 读取所有层
    layers = {name: ds.layers[name][:] for name in ds.layers.keys()}
    if "" in layers:
        main_matrix = layers[""]
    else:
        main_matrix = list(layers.values())[0]
    # 创建新 loom 文件
    loompy.create(output_loom, main_matrix, row_attrs, col_attrs)
    # 写入其他层
    with loompy.connect(output_loom, mode="r+") as ds_out:
        for name, matrix in layers.items():
            if name == "":   # 主层已经写过了，跳过
                continue
            ds_out.layers[name] = matrix
    
    print("所有层已写入:", list(loompy.connect(output_loom).layers.keys()))
print(f"\n完成！新文件保存为：{output_loom}")

原始 loom 形状: (62703, 7375)
可用层: ['', 'mature', 'nascent']
行属性键名: ['target_name']
当前前8个基因（Ensembl ID 等）: ['ENSG00000160072.20' 'ENSG00000279928.2' 'ENSG00000228037.1'
 'ENSG00000142611.17' 'ENSG00000284616.1' 'ENSG00000157911.11'
 'ENSG00000269896.2' 'ENSG00000228463.10']
未匹配的基因数量：38110 / 62703 (60.78%)
替换后前8个基因（gene symbol）: ['ENSG00000160072.20' 'DDX11L17' 'ENSG00000228037.1' 'PRDM16'
 'ENSG00000284616.1' 'ENSG00000157911.11' 'ENSG00000269896.2'
 'ENSG00000228463.10']
所有层已写入: ['', 'mature', 'nascent']

完成！新文件保存为：/data/wuwq/noise/Monod_FLU/symbol_loom/covid11_s20.loom


### 细胞类型数据分割

In [47]:
import loompy
import numpy as np
import pandas as pd
import os

In [269]:
# Step 1-1: 读取细胞类型注释表,创建 barcode -> cell_type 的字典
annotation_file = "/data2/wuwq/noise/monod_FLU/loom/flu_merged_celltype.csv"  # 此表固定不需要变 第一列是 barcode，第二列是 cell_type
barcode_col = "barcode"      # 注释表中 barcode 那一列的列名
celltype_col = "merged_cell_type"   # 注释表中细胞类型那一列的列名

In [418]:
# Step 1-2: 改细胞类型 CD4+ T cell, CD8+ T cell, Monocyte, NK cell, Dendritic cell, B cell (6个细胞类型,无otherT)
target_celltype = "B cell"     # 目标细胞类型   
anno_df = pd.read_csv(annotation_file)  # 如果是 tsv，用 sep="\t"
anno_df[barcode_col] = anno_df[barcode_col].str.strip() #去掉可能的空格
anno_df[celltype_col] = anno_df[celltype_col].str.strip() #去掉可能的空格
barcode_to_type = dict(zip(anno_df[barcode_col], anno_df[celltype_col])) # 创建 barcode -> cell_type 的字典

In [419]:
# Step 2: 连接原始 loom 文件，读取 barcode 并匹配
input_loom = "/data/wuwq/noise/Monod_FLU/symbol_loom/covid11_s20.loom"
with loompy.connect(input_loom) as ds:
    # 请确认这里的键名是否真的是 "barcode"，如果不是请修改
    loom_barcodes = ds.ca["barcode"]     
    # 对 loom 中的每个 barcode 加上 "-**" 后缀后再去注释字典中查找
    cell_types = np.array([
        barcode_to_type.get(bc + "-20", "Unknown") for bc in loom_barcodes  #改此处前缀
    ])  
    selected_cells = np.where(cell_types == target_celltype)[0]  
    print(f"总细胞数: {ds.shape[1]}")
    print(f"匹配到注释的细胞数: {np.sum(cell_types != 'Unknown')}")
    print(f"细胞数: {len(selected_cells)}")
    if len(selected_cells) == 0:
        raise ValueError("没有找到任何细胞!")

总细胞数: 7375
匹配到注释的细胞数: 3749
细胞数: 593


In [420]:
# Step 3: 创建只包含 **细胞类型 的新 loom 文件-- CD4, CD8, MNP, NK, DC, B
output_loom = "/data/wuwq/noise/Monod_FLU/symbol_loom/B/covid11.loom"
selected_cells_idx = np.asarray(selected_cells)  # 必须是 column indices 的 numpy array
layers_to_copy = ["", "mature", "nascent"]
# 第一步：先连接原始文件，检查并获取实际存在的 layers
with loompy.new(output_loom) as dsout:  # 创建空的输出文件
    with loompy.connect(input_loom) as ds:
        # 使用 scan 逐块添加选定细胞的数据和属性
        # layers 参数限制只复制指定的层（提高效率，避免复制不需要的层）
        for (ix, selection, view) in ds.scan(items=selected_cells_idx, axis=1, layers=layers_to_copy):
            # view.layers: 选定层的子矩阵
            # view.ca: 选定细胞的列属性（自动子集）
            # view.ra: 完整的行属性（基因属性，不变）
            dsout.add_columns(view.layers, col_attrs=view.ca, row_attrs=view.ra)